# Comparing filters — exactness, value of switching, multivariate

Building on `01_filtering_quickstart.ipynb`, this notebook compares the fast
exact filter (NGH-MSM-KF) against reference filters:

- the **brute-force** $K^N$ Bayesian filter (ground truth),
- a **pairwise Kalman filter** on the regime-averaged model — the "ignore the
  switching" baseline (a Kalman *couple*, not a traditional reduced filter),
- an **oracle** Kalman filter that knows the true regimes,
- the general **IMM** recursion (`mode="gpb2"`).

Filtering only — no parameter learning.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from prg.classes.GSSSimulator import GSSSimulator
from prg.experiments.models_paper import get_params
from prg.experiments.reference_filters import (
    exact_mixture_filter,
    oracle_filter,
    single_kalman_filter,
    with_stationary_init,
)
from prg.experiments.run_simulations import _params_from_dict
from prg.experiments.study import (
    contrasted_model,  # a "quiet/volatile" model where switching matters
)
from prg.filter.gss_filter import GSSFilter


def build_model(name):
    return with_stationary_init(_params_from_dict(get_params(name)))


def simulate(params, N, seed):
    rs, xs, ys = [], [], []
    for n, r, x, y in GSSSimulator(params, N=N, seed=seed):
        rs.append(int(r))
        xs.append(np.ravel(x))
        ys.append(np.ravel(y))
    return np.array(rs), np.array(xs), np.array(ys)


def run_filter(params, ys, mode="ngh_kf"):
    filt = GSSFilter(params, mode=mode)
    Ex, Var, Pi = [], [], []
    for y in ys:
        res = filt.step(np.asarray(y, float).reshape(params.s, 1))
        Ex.append(res.E_x.ravel())
        Var.append(np.asarray(res.Var_x))
        Pi.append(np.ravel(res.pi))
    return np.array(Ex), np.array(Var), np.array(Pi)


def rmse(est, truth):
    return float(np.sqrt(np.mean((est - truth) ** 2)))

## A. Exactness across the three paper models

For each model we compare NGH-MSM-KF with the brute-force $K^N$ filter on a short
sequence. The maximum disagreement sits at floating-point round-off.

In [ ]:
for name in ["M1", "M2", "M3"]:
    p = build_model(name)
    N = 8 if p.K == 2 else 7
    _, _, ys = simulate(p, N=N, seed=11)
    Ex_ngh, _, _ = run_filter(p, ys)
    Ex_exact, _, _ = exact_mixture_filter(p, ys)
    print(
        f"{name}: K={p.K} q={p.q} s={p.s} | "
        f"max|NGH-MSM-KF - exact| = {np.max(np.abs(Ex_ngh - Ex_exact)):.1e}"
    )

## B. The value of modelling the switching

On a "quiet/volatile" model where the regimes genuinely differ (the $X\!-\!Y$
coupling flips sign across regimes), we compare four estimators by state RMSE.
The exact switching filter recovers most of the gain available to the
regime-aware oracle, while the regime-blind pairwise Kalman cannot.

In [ ]:
pc = with_stationary_init(contrasted_model(m=0.8))
rs, xs, ys = simulate(pc, N=600, seed=0)

Ex_ngh, _, Pi = run_filter(pc, ys)
Ex_kal, _ = single_kalman_filter(pc, ys)
Ex_ora, _ = oracle_filter(pc, rs, ys)

scores = {
    "zero\npredictor": rmse(np.zeros_like(xs), xs),
    "pairwise Kalman\n(no switching)": rmse(Ex_kal, xs),
    "NGH-MSM-KF\n(proposed)": rmse(Ex_ngh, xs),
    "oracle\n(regimes known)": rmse(Ex_ora, xs),
}
print("regime accuracy of NGH-MSM-KF:", f"{np.mean(Pi.argmax(1) == rs):.0%}")

fig, ax = plt.subplots(figsize=(6, 3.3))
ax.bar(range(4), list(scores.values()), color=["#bbbbbb", "#2ca02c", "#1f77b4", "#9467bd"])
ax.set_xticks(range(4))
ax.set_xticklabels(scores.keys(), fontsize=8)
ax.set_ylabel("state RMSE")
ax.set_title("Value of modelling the switching")
for i, v in enumerate(scores.values()):
    ax.text(i, v + 0.006, f"{v:.3f}", ha="center", fontsize=8)
fig.tight_layout()

## C. Multivariate tracking (M2, $q=s=2$)

The same `step` API works unchanged on vector models. Here the 2-D hidden state
is tracked with a calibrated $\pm2\sigma$ band per component.

In [ ]:
p2 = build_model("M2")
rs2, xs2, ys2 = simulate(p2, N=200, seed=5)
Ex2, Var2, Pi2 = run_filter(p2, ys2)
sig2 = np.sqrt(np.clip(np.stack([np.diag(V) for V in Var2]), 0, None))

T = 120
n = np.arange(T)
fig, axes = plt.subplots(p2.q, 1, figsize=(8, 4), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(n, xs2[:T, i], "k.", ms=3, label="true")
    ax.plot(n, Ex2[:T, i], color="#1f77b4", lw=1.2, label=r"$E[X|y]$")
    ax.fill_between(
        n, Ex2[:T, i] - 2 * sig2[:T, i], Ex2[:T, i] + 2 * sig2[:T, i], color="#1f77b4", alpha=0.2
    )
    ax.set_ylabel(f"$X_{i + 1}$")
axes[0].legend(fontsize=8, ncol=2)
axes[-1].set_xlabel("$n$")
axes[0].set_title(r"M2 — multivariate filtered state ($\pm2\sigma$)")
fig.tight_layout()
print("RMSE per component:", np.round(np.sqrt(np.mean((Ex2 - xs2) ** 2, axis=0)), 3))

## D. NGH-MSM-KF vs the general IMM

On this family the general IMM recursion (`mode="gpb2"`) and NGH-MSM-KF
return the **same** answer — both are exact — but NGH-MSM-KF reaches it with a
constant-gain read-out (no per-step Riccati propagation), which is why it is
faster (see experiment `E2` in the report).

In [ ]:
p1 = build_model("M1")
_, _, ys1 = simulate(p1, N=8, seed=3)
Ex_ab, _, _ = run_filter(p1, ys1, mode="ngh_kf")
Ex_imm, _, _ = run_filter(p1, ys1, mode="gpb2")
print(
    "max |NGH-MSM-KF - IMM| on E[X|y]:",
    f"{np.max(np.abs(Ex_ab - Ex_imm)):.1e}   (both exact on the family)",
)

## Recap

- **Exact**: NGH-MSM-KF matches the brute-force $K^N$ filter to round-off (A).
- **Valuable**: it recovers most of the oracle-achievable gain when the regimes
  matter, unlike the regime-blind pairwise Kalman (B).
- **General**: the same `step` API handles multivariate / under-observed models (C).
- **Fast**: it ties the general IMM but with no Riccati propagation (D).

The full simulation study (E1–E10) is in
`docs/rapport_experimental/`, produced by
`python -m prg.experiments.study`.